In [ ]:
#First mount Drive for the dataset if it is not mounted or session is restarted.
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
#Clone or pull the latest code
import os
REPO = "LaneDetection"
if not os.path.exists(f"/content/{REPO}"):
    !git clone https://github.com/abdullahtapanci/LaneDetection.git /content/{REPO}
else:
    !cd /content/{REPO} && git pull

%cd /content/{REPO}

In [ ]:
import os, sys, time
import torch
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
import src.config as cfg
from src.data.dataset import LaneDataset
from src.data.transforms import transform_image
from src.models.lanenet import LaneNet
from src.loss import compute_loss
from src.utils import set_seed, save_checkpoint, binary_iou

In [ ]:
set_seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

In [ ]:
train_ds = LaneDataset(f"{cfg.ROOT_DIR}/train.txt", cfg.ROOT_DIR, transform=transform_image)
val_ds   = LaneDataset(f"{cfg.ROOT_DIR}/val.txt",   cfg.ROOT_DIR, transform=transform_image)

train_loader = DataLoader(train_ds, batch_size=cfg.BATCH_SIZE, shuffle=True,
                          num_workers=2, pin_memory=True, drop_last=True)
val_loader   = DataLoader(val_ds,   batch_size=cfg.BATCH_SIZE, shuffle=False,
                          num_workers=2, pin_memory=True)

print(f"train batches: {len(train_loader)}   val batches: {len(val_loader)}")

In [ ]:
model = LaneNet(embedding_dim=4).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=cfg.LEARNING_RATE, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=cfg.EPOCHS)
class_weights = cfg.CLASS_WEIGHTS.to(device)

print(f"params: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
def train_one_epoch(model, loader, optimizer, class_weights, device):
    model.train()
    sums = {'total': 0., 'binary': 0., 'disc': 0., 'variance': 0., 'distance': 0., 'reg': 0.}
    n = 0

    pbar = tqdm(loader, desc="train", leave=False)
    for img, bin_mask, inst_mask in pbar:
        img       = img.to(device, non_blocking=True)
        bin_mask  = bin_mask.to(device, non_blocking=True)
        inst_mask = inst_mask.to(device, non_blocking=True).long()

        binary_logits, embedding = model(img)
        loss, comps = compute_loss(binary_logits, embedding,
                                   bin_mask, inst_mask,
                                   class_weights=class_weights)

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()

        for k in sums: sums[k] += comps[k]
        n += 1
        pbar.set_postfix({k: f"{v/n:.3f}" for k, v in sums.items()})

    return {k: v / n for k, v in sums.items()}

In [ ]:
@torch.no_grad()
def validate(model, loader, class_weights, device):
    model.eval()
    sums = {'total': 0., 'binary': 0., 'disc': 0.}
    iou_sum = 0.
    n = 0

    for img, bin_mask, inst_mask in tqdm(loader, desc="val", leave=False):
        img       = img.to(device, non_blocking=True)
        bin_mask  = bin_mask.to(device, non_blocking=True)
        inst_mask = inst_mask.to(device, non_blocking=True).long()

        binary_logits, embedding = model(img)
        _, comps = compute_loss(binary_logits, embedding,
                                bin_mask, inst_mask,
                                class_weights=class_weights)

        sums['total']  += comps['total']
        sums['binary'] += comps['binary']
        sums['disc']   += comps['disc']
        iou_sum        += binary_iou(binary_logits, bin_mask)
        n += 1

    return {k: v / n for k, v in sums.items()}, iou_sum / n

In [ ]:
EPOCHS = cfg.EPOCHS
CKPT_DIR = "/content/drive/MyDrive/Lane_Detection_Project/checkpoints"
os.makedirs(CKPT_DIR, exist_ok=True)

best_iou = 0.0
history = []

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    train_metrics = train_one_epoch(model, train_loader, optimizer, class_weights, device)
    val_metrics, val_iou = validate(model, val_loader, class_weights, device)
    scheduler.step()

    elapsed = time.time() - t0
    print(f"[{epoch:02d}/{EPOCHS}] "
          f"train: total={train_metrics['total']:.3f} bin={train_metrics['binary']:.3f} disc={train_metrics['disc']:.3f}  "
          f"val: total={val_metrics['total']:.3f} bin={val_metrics['binary']:.3f} iou={val_iou:.3f}  "
          f"({elapsed:.0f}s)")

    history.append({'epoch': epoch, **{f'train_{k}': v for k, v in train_metrics.items()},
                    **{f'val_{k}': v for k, v in val_metrics.items()}, 'val_iou': val_iou})
    import json
    with open(f"{CKPT_DIR}/history.json", "w") as f:
        json.dump(history, f, indent=2)

    # Save checkpoint each epoch (Drive)
    save_checkpoint(model, optimizer, epoch, f"{CKPT_DIR}/last.pt")
    if val_iou > best_iou:
        best_iou = val_iou
        save_checkpoint(model, optimizer, epoch, f"{CKPT_DIR}/best.pt")
        print(f"  ↑ new best IoU: {best_iou:.4f}")